# Visualizzatore interattivo dei landmark 3D (ATHENA)

Esplora i tracciati 3D prodotti da `athena_analysis.py`: scegli **registrazione**, **landmark**, **grandezza** e
**intervallo di tempo**, e vedi subito quali dati sono **triangolati**, **interpolati** o **mancanti**.

**Come si usa**
1. Kernel **Python (athena)**:
   - in VSCode, *Select Kernel* → *Jupyter Kernel* → `Python (athena)`;
   - oppure da terminale: `~/miniforge3/envs/athena/bin/jupyter lab notebooks/landmarks3d_viewer.ipynb`.
2. Se serve, modifica la cella **Parametri**, poi *Run All*.
3. Usa i controlli dell'ultima sezione. Le funzioni si possono chiamare anche da codice: vedi *Uso da codice* in fondo.

**Qualità di ogni punto** (file `analysis/<trial>/<trial>_3Dquality.npz`, scritto da `athena_analysis.py`):

| stato | significato |
|---|---|
| 🟩 triangolato | visto da ≥ 2 camere in quel frame: misurato (poi filtrato dallo smoothing ATHENA) |
| 🟧 interpolato | visto da < 2 camere: il valore è stato riempito dallo smoothing ATHENA (buchi < 5 frame) |
| 🟥 mancante | nessun valore nel 3D finale (buco ≥ 5 frame) |

## Parametri

In [1]:
# =========================== PARAMETRI (modifica qui) ===========================
from pathlib import Path

# cartella del progetto (contiene scripts/athena_analysis.py): trovata automaticamente
PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'athena_analysis.py').exists()), None)
assert PROJECT is not None, 'apri il notebook dalla cartella del progetto DEXTER_kinematics (o dalle sue sottocartelle)'

DATA_DIR   = PROJECT / 'data'        # cartella con le registrazioni (es. Path.home() / 'remote_data' con SSHFS)
EXPORT_DIR = PROJECT / 'exports'     # dove vengono salvati grafici (HTML) e dati (CSV)

DEFAULT_RECORDING = 'finertapping'   # testo contenuto nel nome della registrazione (None = la prima)
DEFAULT_GROUP     = 'mano destra'    # 'mano destra', 'mano sinistra', 'corpo', 'tutti'
DEFAULT_LANDMARKS = ['R_thumb_tip', 'R_index_tip']
DEFAULT_QUANTITY  = 'posizione'      # una di QUANTITIES (cella successiva)
DEFAULT_AXES      = 'xyz'            # assi mostrati per posizione / velocità per componenti
DEFAULT_ABC       = ('R_thumb_tip', 'R_index_tip', 'R_index_mcp')  # landmark A, B, C per distanza A–B e angolo A–B–C

SMOOTH_FRAMES   = 1      # media mobile sulle posizioni prima del calcolo (1 = nessuna)
SHOW_QUALITY    = True   # bande arancioni = interpolato, rosse = mancante
SHOW_RAW        = False  # sovrappone la triangolazione grezza (prima dello smoothing ATHENA), solo per la posizione
PLOT_HEIGHT     = 650    # altezza dei grafici [px]
MAX_SHADED_RUNS = 400    # oltre questo numero di buchi le bande non vengono disegnate (grafico lento)
FPS_FALLBACK    = 40.0   # usato solo se non si trova il frame rate

## Funzioni (caricamento, calcolo, grafici)

In [2]:
import json, re, sys, warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as W
from IPython.display import display, clear_output

sys.path.insert(0, str(PROJECT / 'scripts'))
import athena_analysis as AA   # nomi dei landmark, ossa, triangolazione grezza

warnings.filterwarnings('ignore', category=RuntimeWarning)

LANDMARKS = list(AA.LANDMARK_NAMES)          # 0-32 body_*, 33-53 R_*, 54-74 L_*
IDX = {n: i for i, n in enumerate(LANDMARKS)}
GROUPS = {
    'mano destra':   [n for n in LANDMARKS if n.startswith('R_')],
    'mano sinistra': [n for n in LANDMARKS if n.startswith('L_')],
    'corpo':         [n for n in LANDMARKS if n.startswith('body_')],
    'tutti':         LANDMARKS,
}
STATUS = {0: 'mancante', 1: 'interpolato', 2: 'triangolato'}
STATUS_COLOR = {0: '#d62728', 1: '#ff7f0e', 2: '#2ca02c'}
QUANTITIES = ['posizione', 'velocità (modulo)', 'velocità (componenti)', 'accelerazione (modulo)',
              'distanza A–B', 'angolo A–B–C', 'numero di camere']
UNITS = {'posizione': 'mm', 'velocità (modulo)': 'mm/s', 'velocità (componenti)': 'mm/s',
         'accelerazione (modulo)': 'mm/s²', 'distanza A–B': 'mm', 'angolo A–B–C': '°', 'numero di camere': 'camere'}
COLORS = px.colors.qualitative.Plotly + px.colors.qualitative.Dark24
SKELETON = ([(33 + a, 33 + b) for a, b in AA.HAND_BONES] + [(54 + a, 54 + b) for a, b in AA.HAND_BONES]
            + [(11, 12), (11, 13), (12, 14), (13, 54), (14, 33), (11, 23), (12, 24), (23, 24)])


# ------------------------------------------------------------------ caricamento
def find_trials(data_dir=None):
    """{nome: (main folder ATHENA, trial)} per ogni 3D disponibile in data_dir."""
    data_dir = Path(data_dir or DATA_DIR).expanduser()
    files = []
    for pat in ('*/athena/landmarks/*/*_3Dlandmarks.npy', 'athena/landmarks/*/*_3Dlandmarks.npy',
                'landmarks/*/*_3Dlandmarks.npy'):
        files += sorted(data_dir.glob(pat))
    found = {}
    for f in files:
        main = f.parent.parent.parent
        rec = main.parent if main.name == 'athena' else main
        found.setdefault(rec.name, []).append((main, f.parent.name))
    return {(rec if len(items) == 1 else f'{rec} / {trial}'): (main, trial)
            for rec, items in found.items() for main, trial in items}


_CACHE = {}

def load_trial(name, data_dir=None):
    """Carica 3D finale, triangolazione grezza, stato di qualità e camere per punto.
    `name`: nome completo o parte del nome della registrazione."""
    trials = find_trials(data_dir)
    if name not in trials:
        hits = [k for k in trials if name.lower() in k.lower()]
        if len(hits) != 1:
            raise ValueError(f'"{name}": {len(hits)} registrazioni corrispondenti {hits[:5]}')
        name = hits[0]
    if name in _CACHE:
        return _CACHE[name]
    main, trial = trials[name]
    d3 = np.load(main / 'landmarks' / trial / f'{trial}_3Dlandmarks.npy')
    adir = main / 'analysis' / trial
    fps = None
    if (adir / 'run_info.json').exists():
        fps = float(json.loads((adir / 'run_info.json').read_text()).get('fps') or 0) or None
    if fps is None:
        try:
            fps = AA.video_frames(sorted((main / 'videos' / trial).glob('*.mp4'))[0])[1]
        except Exception:
            fps = FPS_FALLBACK
    qf = adir / f'{trial}_3Dquality.npz'
    if qf.exists():
        q = np.load(qf)
        status, ncams, raw = q['status'], q['ncams'], q['raw3d'].astype(float)
    else:  # registrazioni elaborate prima dell'introduzione del file di qualità
        print(f'{name}: file di qualità assente, calcolo la triangolazione grezza ...')
        raw, _, ncams = AA.triangulate_raw(main, trial)
        status = AA.point_status(d3, raw)
        if adir.is_dir():
            AA.save_quality(adir, trial, d3, raw, ncams)
    tr = dict(name=name, main=main, trial=trial, d3=d3, raw=raw, status=status, ncams=ncams,
              fps=fps, t=np.arange(len(d3)) / fps)
    _CACHE[name] = tr
    return tr


# ------------------------------------------------------------------ calcolo
def frame_slice(tr, t_range=None):
    n = len(tr['t'])
    if t_range is None:
        return slice(0, n)
    a = max(0, int(np.floor(t_range[0] * tr['fps'] + 1e-6)))
    b = min(n, int(np.floor(t_range[1] * tr['fps'] + 1e-6)) + 1)
    return slice(a, max(b, a + 1))


def moving_avg(y, n):
    """Media mobile centrata su n frame (i NaN restano NaN)."""
    if n <= 1:
        return y
    sm = pd.DataFrame(y.reshape(len(y), -1)).rolling(n, center=True, min_periods=1).mean().to_numpy()
    sm = np.array(sm, dtype=float).reshape(y.shape)  # copia scrivibile
    sm[np.isnan(y)] = np.nan
    return sm


def runs(mask):
    """[(inizio, fine_esclusa)] dei tratti True."""
    d = np.diff(np.concatenate([[0], np.asarray(mask, int), [0]]))
    return list(zip(np.where(d == 1)[0], np.where(d == -1)[0]))


def compute_series(tr, landmarks, quantity='posizione', axes='xyz', A=None, B=None, C=None, t_range=None, smooth=1):
    """Calcola la grandezza richiesta nell'intervallo.
    Ritorna (DataFrame[frame, time_s, <serie>..., stato_<landmark>...], colonne delle serie, landmark coinvolti)."""
    sl = frame_slice(tr, t_range)
    d3, fps = tr['d3'], tr['fps']
    df = pd.DataFrame({'frame': np.arange(len(d3))[sl], 'time_s': tr['t'][sl]})
    pos = lambda n: moving_avg(d3[:, IDX[n]], smooth)
    ax = {'x': 0, 'y': 1, 'z': 2}
    cols = []
    if quantity == 'distanza A–B':
        involved = [A, B]
        cols = [f'dist {A}–{B}']
        df[cols[0]] = np.linalg.norm(pos(A) - pos(B), axis=1)[sl]
    elif quantity == 'angolo A–B–C':
        involved = [A, B, C]
        u, w = pos(A) - pos(B), pos(C) - pos(B)
        cos = (u * w).sum(1) / (np.linalg.norm(u, axis=1) * np.linalg.norm(w, axis=1))
        cols = [f'angolo {A}–{B}–{C}']
        df[cols[0]] = np.degrees(np.arccos(np.clip(cos, -1, 1)))[sl]
    else:
        involved = list(landmarks)
        for n in involved:
            if quantity == 'posizione':
                p = pos(n)
                for a in axes:
                    cols.append(f'{n}_{a}'); df[cols[-1]] = p[sl, ax[a]]
            elif quantity == 'numero di camere':
                cols.append(f'{n}_ncams'); df[cols[-1]] = tr['ncams'][sl, IDX[n]]
            else:
                v = np.gradient(pos(n), axis=0) * fps
                if quantity == 'accelerazione (modulo)':
                    cols.append(f'{n}_|a|'); df[cols[-1]] = np.linalg.norm(np.gradient(moving_avg(v, smooth), axis=0) * fps, axis=1)[sl]
                elif quantity == 'velocità (componenti)':
                    for a in axes:
                        cols.append(f'{n}_v{a}'); df[cols[-1]] = v[sl, ax[a]]
                else:
                    cols.append(f'{n}_|v|'); df[cols[-1]] = np.linalg.norm(v, axis=1)[sl]
    for n in involved:
        df[f'stato_{n}'] = pd.Series(tr['status'][sl, IDX[n]]).map(STATUS).to_numpy()
    return df, cols, involved


def quality_counts(tr, landmarks, t_range=None):
    s = tr['status'][frame_slice(tr, t_range)][:, [IDX[n] for n in landmarks]]
    return {STATUS[k]: 100 * float((s == k).mean()) for k in (2, 1, 0)}


# ------------------------------------------------------------------ grafici
def _empty(msg):
    fig = go.Figure()
    fig.add_annotation(text=msg, showarrow=False, font=dict(size=16))
    fig.update_layout(height=250, xaxis_visible=False, yaxis_visible=False)
    return fig


def _shade_quality(fig, tr, sl, involved):
    """Bande: arancione = almeno un landmark interpolato, rosso = almeno uno mancante."""
    worst = tr['status'][sl][:, [IDX[n] for n in involved]].min(1)
    t, h = tr['t'][sl], 0.5 / tr['fps']
    for code, label in ((1, 'interpolato'), (0, 'mancante')):
        rr = runs(worst == code)
        if not rr:
            continue
        fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers', name=f'{label} ({len(rr)} tratti)',
                                 marker=dict(symbol='square', size=12, color=STATUS_COLOR[code], opacity=0.4)))
        if len(rr) > MAX_SHADED_RUNS:
            continue
        for a, b in rr:
            fig.add_vrect(x0=t[a] - h, x1=t[b - 1] + h, fillcolor=STATUS_COLOR[code], opacity=0.18,
                          line_width=0, layer='below', row='all', col=1)


def timeseries_figure(tr, landmarks, quantity='posizione', axes='xyz', A=None, B=None, C=None, t_range=None,
                      smooth=1, show_quality=True, show_raw=False, height=None):
    """Serie temporali della grandezza scelta; posizione e velocità per componenti: un pannello per asse."""
    A, B, C = A or DEFAULT_ABC[0], B or DEFAULT_ABC[1], C or DEFAULT_ABC[2]
    axes = axes or 'xyz'
    df, cols, involved = compute_series(tr, landmarks, quantity, axes, A, B, C, t_range, smooth)
    if not cols:
        return _empty('Seleziona almeno un landmark')
    sl = frame_slice(tr, t_range)
    by_axis = quantity in ('posizione', 'velocità (componenti)')
    nrows = len(axes) if by_axis else 1
    unit = UNITS[quantity]
    fig = make_subplots(rows=nrows, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                        subplot_titles=[f'{a}  [{unit}]' for a in axes] if by_axis and nrows > 1 else None)
    seen = set()
    for col in cols:
        lm = next((n for n in involved if col.startswith(n + '_')), None)
        key = lm or col
        color = COLORS[(involved.index(lm) if lm else 0) % len(COLORS)]
        row = axes.index(col[-1]) + 1 if by_axis else 1
        st = df[f'stato_{lm}'] if lm else pd.DataFrame({n: df[f'stato_{n}'] for n in involved}).apply(
            lambda r: min(r, key=lambda s: [k for k, v in STATUS.items() if v == s][0]), axis=1)
        fig.add_trace(go.Scattergl(
            x=df.time_s, y=df[col], mode='lines', name=key, legendgroup=key, showlegend=key not in seen,
            line=dict(color=color, width=1.6), customdata=np.stack([df.frame, st], -1),
            hovertemplate=f'<b>{col}</b><br>t = %{{x:.3f}} s (frame %{{customdata[0]}})<br>'
                          f'%{{y:.2f}} {unit}<br>%{{customdata[1]}}<extra></extra>'), row=row, col=1)
        seen.add(key)
        if show_raw and quantity == 'posizione' and lm:
            fig.add_trace(go.Scattergl(
                x=df.time_s, y=tr['raw'][sl, IDX[lm], 'xyz'.index(col[-1])], mode='markers',
                name=f'{lm} grezzo', legendgroup=key, showlegend=False, marker=dict(color=color, size=3, opacity=0.5),
                hovertemplate=f'{col} grezzo: %{{y:.2f}} {unit}<extra></extra>'), row=row, col=1)
    if show_quality:
        _shade_quality(fig, tr, sl, involved)
    q = quality_counts(tr, involved, t_range)
    fig.update_layout(
        height=height or PLOT_HEIGHT, hovermode='closest', margin=dict(t=70, l=60, r=20, b=40),
        title=dict(text=f'<b>{tr["name"]}</b> – {quantity}<br><sup>{df.time_s.iloc[0]:.2f}–{df.time_s.iloc[-1]:.2f} s, '
                        f'{len(df)} frame  |  ' + ', '.join(f'{k} {v:.1f}%' for k, v in q.items()) + '</sup>'))
    fig.update_xaxes(title_text='tempo [s]', row=nrows, col=1)
    if not by_axis or nrows == 1:
        fig.update_yaxes(title_text=f'{quantity} [{unit}]' + (f' – {axes}' if by_axis else ''))
    return fig


def trajectory_figure(tr, landmarks, t_range=None, frame=None, skeleton=True, height=None):
    """Traiettorie 3D nell'intervallo (colore = tempo) + scheletro delle mani al frame scelto."""
    sl = frame_slice(tr, t_range)
    t = tr['t'][sl]
    fig = go.Figure()
    for i, n in enumerate(landmarks):
        p = tr['d3'][sl, IDX[n]]
        fig.add_trace(go.Scatter3d(
            x=p[:, 0], y=p[:, 1], z=p[:, 2], mode='lines+markers', name=n,
            line=dict(color=COLORS[i % len(COLORS)], width=3),
            marker=dict(size=2.5, color=t, colorscale='Viridis', showscale=(i == 0),
                        colorbar=dict(title='t [s]', len=0.6) if i == 0 else None),
            customdata=t, hovertemplate=f'{n}<br>t=%{{customdata:.3f}} s<br>x=%{{x:.1f}} y=%{{y:.1f}} z=%{{z:.1f}}<extra></extra>'))
    if skeleton and frame is not None:
        P = tr['d3'][frame]
        xs, ys, zs = [], [], []
        for a, b in SKELETON:
            xs += [P[a, 0], P[b, 0], None]; ys += [P[a, 1], P[b, 1], None]; zs += [P[a, 2], P[b, 2], None]
        fig.add_trace(go.Scatter3d(x=xs, y=ys, z=zs, mode='lines', name=f'scheletro frame {frame}',
                                   line=dict(color='gray', width=5), hoverinfo='skip'))
        hands = list(range(33, 75))
        st = tr['status'][frame, hands]
        fig.add_trace(go.Scatter3d(x=P[hands, 0], y=P[hands, 1], z=P[hands, 2], mode='markers', name='giunti (stato)',
                                   marker=dict(size=4, color=[STATUS_COLOR[s] for s in st]),
                                   text=[f'{LANDMARKS[h]}: {STATUS[s]}, {tr["ncams"][frame, h]} camere' for h, s in zip(hands, st)],
                                   hoverinfo='text'))
    if not fig.data:
        return _empty('Seleziona almeno un landmark')
    fig.update_layout(height=height or PLOT_HEIGHT, margin=dict(t=50, l=0, r=0, b=0),
                      title=f'<b>{tr["name"]}</b> – traiettorie 3D {t[0]:.2f}–{t[-1]:.2f} s'
                            + (f', scheletro al frame {frame} ({frame / tr["fps"]:.2f} s)' if frame is not None else ''),
                      scene=dict(aspectmode='data', xaxis_title='X [mm]', yaxis_title='Y [mm]', zaxis_title='Z [mm]'))
    return fig


def quality_heatmap(tr, landmarks, t_range=None, what='stato', height=None):
    """Mappa nel tempo dello stato (triangolato / interpolato / mancante) o del numero di camere per landmark."""
    if not landmarks:
        return _empty('Seleziona almeno un landmark')
    sl = frame_slice(tr, t_range)
    idx = [IDX[n] for n in landmarks]
    t = tr['t'][sl]
    if what == 'stato':
        z = tr['status'][sl][:, idx].T
        cs = [[0, STATUS_COLOR[0]], [1 / 3, STATUS_COLOR[0]], [1 / 3, STATUS_COLOR[1]], [2 / 3, STATUS_COLOR[1]],
              [2 / 3, STATUS_COLOR[2]], [1, STATUS_COLOR[2]]]
        hm = go.Heatmap(z=z, x=t, y=landmarks, zmin=-0.5, zmax=2.5, colorscale=cs,
                        colorbar=dict(tickvals=[0, 1, 2], ticktext=[STATUS[k] for k in (0, 1, 2)]),
                        hovertemplate='%{y}<br>t=%{x:.3f} s<extra></extra>')
    else:
        z = tr['ncams'][sl][:, idx].T
        hm = go.Heatmap(z=z, x=t, y=landmarks, colorscale='Viridis', zmin=0, zmax=int(tr['ncams'].max()),
                        colorbar=dict(title='camere'), hovertemplate='%{y}<br>t=%{x:.3f} s<br>%{z} camere<extra></extra>')
    fig = go.Figure(hm)
    fig.update_layout(height=height or max(300, 22 * len(landmarks) + 120), margin=dict(t=50, l=160, r=20, b=40),
                      title=f'<b>{tr["name"]}</b> – {"stato dei punti" if what == "stato" else "camere che vedono il punto"}',
                      xaxis_title='tempo [s]', yaxis=dict(autorange='reversed'))
    return fig


def stats_table(tr, landmarks, t_range=None):
    """Statistiche per landmark nell'intervallo: qualità, camere, buchi, ampiezza del movimento, velocità."""
    sl = frame_slice(tr, t_range)
    fps, rows = tr['fps'], []
    for n in landmarks:
        i = IDX[n]
        s, nc, p = tr['status'][sl, i], tr['ncams'][sl, i], tr['d3'][sl, i]
        v = np.linalg.norm(np.gradient(tr['d3'][:, i], axis=0) * fps, axis=1)[sl]
        gaps = runs(s < 2)
        rows.append({
            'landmark': n,
            '% triangolato': 100 * (s == 2).mean(), '% interpolato': 100 * (s == 1).mean(), '% mancante': 100 * (s == 0).mean(),
            'camere medie': nc[s == 2].mean() if (s == 2).any() else np.nan,
            'camere min': nc[s == 2].min() if (s == 2).any() else np.nan,
            'n buchi': len(gaps), 'buco max [s]': max((b - a for a, b in gaps), default=0) / fps,
            'range x [mm]': np.nanmax(p[:, 0]) - np.nanmin(p[:, 0]) if np.isfinite(p[:, 0]).any() else np.nan,
            'range y [mm]': np.nanmax(p[:, 1]) - np.nanmin(p[:, 1]) if np.isfinite(p[:, 1]).any() else np.nan,
            'range z [mm]': np.nanmax(p[:, 2]) - np.nanmin(p[:, 2]) if np.isfinite(p[:, 2]).any() else np.nan,
            'vel media [mm/s]': np.nanmean(v), 'vel max [mm/s]': np.nanmax(v) if np.isfinite(v).any() else np.nan,
        })
    return pd.DataFrame(rows).set_index('landmark').round(2) if rows else pd.DataFrame()


def safe_name(s):
    return re.sub(r'[^\w.-]+', '_', s).strip('_')


print(f'{len(find_trials())} registrazioni con 3D in {DATA_DIR}')

7 registrazioni con 3D in /home/roberto/Desktop/DEXTER_kinematics/data


## Visualizzatore interattivo

- **Gruppo** filtra l'elenco dei landmark. Per selezionarne più di uno usa Ctrl/Shift + clic.
- **Grandezza**:
  - posizione, velocità (modulo o per componenti), accelerazione;
  - distanza tra due landmark (A–B) e angolo in B (A–B–C), per esempio la flessione delle dita;
  - numero di camere che vedono il punto.
- **Intervallo**: trascina gli estremi oppure premi *Tutto il tracciato*. Nei grafici puoi anche zoomare col mouse.
- **Schede**:
  - *Serie temporali*: le bande arancioni sono interpolate, le rosse mancanti;
  - *Traiettoria 3D*: scheletro al frame scelto, giunti colorati per stato;
  - *Qualità*: mappa nel tempo;
  - *Statistiche*: tabella per landmark.
- **Salva** scrive il grafico (HTML interattivo) o i dati (CSV, con lo stato di ogni punto) in `EXPORT_DIR`.

In [3]:
TRIALS = find_trials()
assert TRIALS, f'nessun 3D trovato in {DATA_DIR}: elabora prima le registrazioni con athena_analysis.py'
names = list(TRIALS)
rec0 = next((n for n in names if DEFAULT_RECORDING and DEFAULT_RECORDING.lower() in n.lower()), names[0])
lay = lambda w: W.Layout(width=w)
sty = dict(description_width='110px')

w_rec = W.Dropdown(options=names, value=rec0, description='Registrazione', layout=lay('560px'), style=sty)
w_group = W.ToggleButtons(options=list(GROUPS), value=DEFAULT_GROUP, description='Gruppo', style=dict(description_width='110px', button_width='110px'))
w_lms = W.SelectMultiple(options=GROUPS[DEFAULT_GROUP], value=[n for n in DEFAULT_LANDMARKS if n in GROUPS[DEFAULT_GROUP]],
                         rows=14, description='Landmark', layout=lay('330px'), style=sty)
w_qty = W.Dropdown(options=QUANTITIES, value=DEFAULT_QUANTITY, description='Grandezza', layout=lay('360px'), style=sty)
w_axes = [W.Checkbox(value=a in DEFAULT_AXES, description=a, indent=False, layout=lay('45px')) for a in 'xyz']
w_axes_box = W.HBox([W.Label('Assi', layout=lay('110px'))] + w_axes)
w_A = W.Dropdown(options=LANDMARKS, value=DEFAULT_ABC[0], description='A', layout=lay('360px'), style=sty)
w_B = W.Dropdown(options=LANDMARKS, value=DEFAULT_ABC[1], description='B', layout=lay('360px'), style=sty)
w_C = W.Dropdown(options=LANDMARKS, value=DEFAULT_ABC[2], description='C (angolo in B)', layout=lay('360px'), style=sty)
w_smooth = W.IntSlider(value=SMOOTH_FRAMES, min=1, max=21, step=2, description='Media mobile', continuous_update=False,
                       layout=lay('360px'), style=sty)
w_quality = W.Checkbox(value=SHOW_QUALITY, description='Evidenzia interpolati / mancanti', indent=False)
w_raw = W.Checkbox(value=SHOW_RAW, description='Mostra triangolazione grezza (posizione)', indent=False)
w_time = W.FloatRangeSlider(min=0, max=1, step=0.025, value=(0, 1), description='Intervallo [s]', continuous_update=False,
                            readout_format='.2f', layout=lay('900px'), style=sty)
w_full = W.Button(description='Tutto il tracciato', icon='arrows-h')
w_frame = W.IntSlider(description='Frame scheletro', continuous_update=False, layout=lay('900px'), style=sty)
w_heat = W.ToggleButtons(options=['stato', 'camere'], description='Mappa di', style=sty)
w_heat_group = W.Checkbox(value=True, description="Tutto il gruppo (non solo i landmark selezionati)", indent=False)
w_save_html = W.Button(description='Salva grafico (HTML)', icon='save')
w_save_csv = W.Button(description='Salva dati (CSV)', icon='table')
w_info, w_msg = W.HTML(), W.HTML()
out_ts, out_3d, out_heat, out_stats = W.Output(), W.Output(), W.Output(), W.Output()
tabs = W.Tab(children=[out_ts, W.VBox([w_frame, out_3d]), W.VBox([W.HBox([w_heat, w_heat_group]), out_heat]), out_stats])
tabs.titles = ('Serie temporali', 'Traiettoria 3D', 'Qualità', 'Statistiche')
STATE = {'busy': False, 'fig': None}


def params():
    return dict(tr=load_trial(w_rec.value), landmarks=list(w_lms.value), quantity=w_qty.value,
                axes=''.join(c.description for c in w_axes if c.value) or 'xyz', A=w_A.value, B=w_B.value, C=w_C.value,
                t_range=w_time.value, smooth=w_smooth.value)


def update_visibility():
    q = w_qty.value
    w_axes_box.layout.display = '' if q in ('posizione', 'velocità (componenti)') else 'none'
    w_A.layout.display = w_B.layout.display = '' if q in ('distanza A–B', 'angolo A–B–C') else 'none'
    w_C.layout.display = '' if q == 'angolo A–B–C' else 'none'
    w_raw.layout.display = '' if q == 'posizione' else 'none'


def redraw(*_):
    if STATE['busy']:
        return
    update_visibility()
    p = params()
    tr = p['tr']
    sl = frame_slice(tr, p['t_range'])
    w_info.value = (f"<b>{tr['name']}</b> &nbsp; {len(tr['t'])} frame @ {tr['fps']:g} fps = {len(tr['t']) / tr['fps']:.2f} s"
                    f" &nbsp;|&nbsp; intervallo: frame {sl.start}–{sl.stop - 1} ({sl.stop - sl.start} frame)")
    i = tabs.selected_index or 0
    try:
        if i == 0:
            fig = timeseries_figure(**p, show_quality=w_quality.value, show_raw=w_raw.value)
            target = out_ts
        elif i == 1:
            fig = trajectory_figure(p['tr'], p['landmarks'], p['t_range'], frame=w_frame.value)
            target = out_3d
        elif i == 2:
            lms = GROUPS[w_group.value] if w_heat_group.value else p['landmarks']
            fig = quality_heatmap(p['tr'], lms, p['t_range'], what=w_heat.value)
            target = out_heat
        else:
            fig = None
            with out_stats:
                clear_output(wait=True)
                display(stats_table(p['tr'], p['landmarks'] or GROUPS[w_group.value], p['t_range']))
        if fig is not None:
            STATE['fig'] = fig
            with target:
                clear_output(wait=True)
                fig.show()
    except Exception as e:
        w_msg.value = f'<span style="color:#d62728">errore: {type(e).__name__}: {e}</span>'
    else:
        if w_msg.value.startswith('<span style="color:#d62728">'):
            w_msg.value = ''


def on_recording(*_):
    tr = load_trial(w_rec.value)
    dur = (len(tr['t']) - 1) / tr['fps']
    STATE['busy'] = True
    w_time.step = 1 / tr['fps']
    w_time.max = dur
    w_time.min = 0
    w_time.value = (0, dur)
    STATE['busy'] = False
    on_time()


def on_time(*_):
    tr = load_trial(w_rec.value)
    sl = frame_slice(tr, w_time.value)
    STATE['busy'] = True
    w_frame.min, w_frame.max = 0, len(tr['t']) - 1
    w_frame.min, w_frame.max = sl.start, sl.stop - 1
    w_frame.value = (sl.start + sl.stop - 1) // 2
    STATE['busy'] = False
    redraw()


def on_group(*_):
    opts = GROUPS[w_group.value]
    keep = [n for n in w_lms.value if n in opts]
    STATE['busy'] = True
    w_lms.options = opts
    w_lms.value = keep or [n for n in DEFAULT_LANDMARKS if n in opts] or opts[:1]
    STATE['busy'] = False
    redraw()


def on_full(_):
    w_time.value = (w_time.min, w_time.max)


def report_errors(fn):
    """Mostra nel notebook gli errori dei pulsanti (altrimenti ipywidgets li nasconde)."""
    def wrapper(*a):
        try:
            fn(*a)
        except Exception as e:
            w_msg.value = f'<span style="color:#d62728">errore: {type(e).__name__}: {e}</span>'
    return wrapper


@report_errors
def save_html(_):
    if STATE['fig'] is None:
        return
    p = params()
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    f = EXPORT_DIR / (safe_name(f"{p['tr']['name']}_{tabs.titles[tabs.selected_index or 0]}_{p['quantity']}_"
                               f"{p['t_range'][0]:.2f}-{p['t_range'][1]:.2f}s") + '.html')
    STATE['fig'].write_html(f, include_plotlyjs='cdn')
    w_msg.value = f'grafico salvato: <code>{f}</code>'


@report_errors
def save_csv(_):
    p = params()
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    df, cols, involved = compute_series(**p)
    f = EXPORT_DIR / (safe_name(f"{p['tr']['name']}_{p['quantity']}_{p['t_range'][0]:.2f}-{p['t_range'][1]:.2f}s") + '.csv')
    df.to_csv(f, index=False)
    w_msg.value = f'dati salvati ({len(df)} righe, {len(cols)} serie + stato per landmark): <code>{f}</code>'


w_rec.observe(on_recording, 'value')
w_group.observe(on_group, 'value')
w_time.observe(on_time, 'value')
for w in (w_lms, w_qty, w_A, w_B, w_C, w_smooth, w_quality, w_raw, w_frame, w_heat, w_heat_group, *w_axes):
    w.observe(redraw, 'value')
tabs.observe(redraw, 'selected_index')
w_full.on_click(on_full)
w_save_html.on_click(save_html)
w_save_csv.on_click(save_csv)

controls = W.HBox([
    W.VBox([w_group, w_lms]),
    W.VBox([w_qty, w_axes_box, w_A, w_B, w_C, w_smooth, w_quality, w_raw]),
])
ui = W.VBox([w_rec, controls, W.HBox([w_time, w_full]), w_info, tabs, W.HBox([w_save_html, w_save_csv]), w_msg])
display(ui)
on_recording()

## Uso da codice (senza widget)

Le stesse funzioni si possono chiamare direttamente. È comodo per figure ripetibili, per un report o per
confrontare più registrazioni.

```python
tr = load_trial('louis_finertapping')               # basta una parte del nome
timeseries_figure(tr, ['R_index_tip'], 'velocità (modulo)', t_range=(2, 8), smooth=3).show()
df, cols, _ = compute_series(tr, [], 'distanza A–B', A='R_thumb_tip', B='R_index_tip')   # DataFrame
```

| funzione | cosa fa |
|---|---|
| `find_trials()` | elenco delle registrazioni con 3D disponibile |
| `load_trial(nome)` | dati: `d3` (frame, 75, 3) mm, `raw`, `status`, `ncams`, `fps`, `t` |
| `compute_series(tr, landmarks, grandezza, ...)` | DataFrame con serie e stato di ogni punto |
| `timeseries_figure(...)`, `trajectory_figure(...)`, `quality_heatmap(...)` | figure plotly |
| `stats_table(tr, landmarks, t_range)` | tabella di qualità e movimento |

In [4]:
# esempio: distanza pollice–indice destro nei primi 10 s di una registrazione, con qualità
tr = load_trial(rec0)
timeseries_figure(tr, [], 'distanza A–B', A='R_thumb_tip', B='R_index_tip', t_range=(0, 10), height=350).show()
stats_table(tr, ['R_wrist', 'R_thumb_tip', 'R_index_tip', 'L_wrist', 'L_thumb_tip', 'L_index_tip'])

,% triangolato,% interpolato,% mancante,camere medie,camere min,n buchi,buco max [s],range x [mm],range y [mm],range z [mm],vel media [mm/s],vel max [mm/s]
landmark,,,,,,,,,,,,
R_wrist,100.0,0.0,0.0,7.75,4,0,0.0,109.78,43.56,55.89,39.03,1289.30
R_thumb_tip,100.0,0.0,0.0,7.75,4,0,0.0,127.76,27.81,119.71,71.62,986.64
R_index_tip,100.0,0.0,0.0,7.75,4,0,0.0,180.75,64.57,113.76,90.85,2055.12
L_wrist,100.0,0.0,0.0,7.86,6,0,0.0,92.62,49.74,76.35,25.97,500.98
L_thumb_tip,100.0,0.0,0.0,7.86,6,0,0.0,96.80,49.46,143.90,81.10,837.34
L_index_tip,100.0,0.0,0.0,7.86,6,0,0.0,93.12,78.30,138.06,84.40,642.59
